In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

from sklearn.metrics import (
    accuracy_score,
    f1_score,
    confusion_matrix
)


In [20]:
df = pd.read_csv('../data/products.csv')

In [3]:
x = df['product']
y = df['category']

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [4]:
y_pred = np.load('../variables/y_pred.npy', allow_pickle=True)
y_pred_char = np.load('../variables/y_pred_char.npy', allow_pickle=True)
y_pred_svm = np.load('../variables/y_pred_svm.npy', allow_pickle=True)
xlmr_results = np.load('../variables/xlmr_results.npy', allow_pickle=True)

Model Comparison

In [12]:
df_language = df.loc[
    x_test.index,
    ['language']
].copy()

model_comparison = pd.DataFrame({
    'Language': df_language['language'],
    'Product': x_test,
    'Actual': y_test,
    'Word TF-IDF LR': y_pred,
    'Char TF-IDF LR': y_pred_char,
    'Char TF-IDF SVM': y_pred_svm,
    'XLM-R': xlmr_results
})

model_comparison

,Language,Product,Actual,Word TF-IDF LR,Char TF-IDF LR,Char TF-IDF SVM,XLM-R
185,zh,便携式投影仪,IT,IT,Electrical,Electrical,IT
33,id,map tulang ukuran A4,Office Supplies,Office Supplies,Safety Equipment,Office Supplies,Office Supplies
58,id,lampu sorot tanam untuk koridor,Electrical,Electrical,Electrical,Electrical,Safety Equipment
37,id,buku agenda ukran A5,Office Supplies,IT,Tools & Hardware,Tools & Hardware,Office Supplies
79,id,baut kecil segi enam,Tools & Hardware,Tools & Hardware,Office Supplies,Tools & Hardware,Office Supplies
124,zh,钢丝钳8寸,Tools & Hardware,IT,Tools & Hardware,Tools & Hardware,IT
103,zh,防降噪耳罩,Safety Equipment,IT,IT,IT,IT
116,zh,复合式洗眼器12L/min,Safety Equipment,IT,Safety Equipment,Safety Equipment,IT
106,zh,安全鞋耐200J冲击,Safety Equipment,IT,Safety Equipment,Safety Equipment,Safety Equipment
101,zh,防护眼镜,Safety Equipment,IT,Safety Equipment,Safety Equipment,Safety Equipment


In [6]:
word_tfidf_lr_accuracy = accuracy_score(
    y_test, y_pred
)

word_tfidf_lr_f1 = f1_score(
    y_test, y_pred,
    average='macro'
)

char_tfidf_lr_accuracy = accuracy_score(
    y_test, y_pred_char
)

char_tfidf_lr_f1 = f1_score(
    y_test, y_pred_char,
    average='macro'
)

char_tfidf_svm_accuracy = accuracy_score(
    y_test, y_pred_svm
)

char_tfidf_svm_f1 = f1_score(
    y_test, y_pred_svm,
    average='macro'
)

xlmr_accuracy = accuracy_score(
    y_test, xlmr_results
)

xlmr_f1 = f1_score(
    y_test, xlmr_results,
    average='macro'
)

In [7]:
score_comparison = pd.DataFrame({
    'Model': [
        'Word TF-IDF LR',
        'Char TF-IDF LR',
        'Char TF-IDF SVM',
        'XLM-R'
    ],
    'Accuracy': [
        word_tfidf_lr_accuracy,
        char_tfidf_lr_accuracy,
        char_tfidf_svm_accuracy,
        xlmr_accuracy
    ],
    'Macro F-1': [
        word_tfidf_lr_f1,
        char_tfidf_lr_f1,
        char_tfidf_svm_f1,
        xlmr_f1
    ]
})

score_comparison

,Model,Accuracy,Macro F-1
0,Word TF-IDF LR,0.400,0.404767
1,Char TF-IDF LR,0.450,0.451573
2,Char TF-IDF SVM,0.475,0.475766
3,XLM-R,0.475,0.374975


Language Comparison

In [27]:
model_cols = [
    'Word TF-IDF LR',
    'Char TF-IDF LR',
    'Char TF-IDF SVM',
    'XLM-R'
]

correct_cols = []

for model_name in model_cols:
    correct_col = f'{model_name} Correct'
    model_comparison[correct_col] = (
        model_comparison[model_name] == model_comparison['Actual']
    )
    correct_cols.append(correct_col)

language_accuracy = (
    model_comparison.groupby('Language')[correct_cols]
    .mean()
)

language_accuracy

,Word TF-IDF LR Correct,Char TF-IDF LR Correct,Char TF-IDF SVM Correct,XLM-R Correct
Language,,,,
id,0.666667,0.444444,0.555556,0.500000
zh,0.181818,0.454545,0.409091,0.454545


In [30]:
language_f1 = {}

for language in ['id', 'zh']:
    subset = model_comparison[model_comparison['Language'] == language]

    for model_name in model_cols:
        language_f1[(language, model_name)] = f1_score(
            subset['Actual'],
            subset[model_name],
            labels=sorted(subset['Actual'].unique()),
            average='macro',
            zero_division=0
        )

language_f1_df = pd.DataFrame(
    language_f1,
    index=['Macro F1']
).T

language_f1_df

Macro F1
id Word TF-IDF LR   0.673853
   Char TF-IDF LR   0.447619
   Char TF-IDF SVM  0.552222
   XLM-R            0.373333
zh Word TF-IDF LR   0.061538
   Char TF-IDF LR   0.433333
   Char TF-IDF SVM  0.372381
   XLM-R            0.405556